# Qdrant Local Embeddings Pipeline

This notebook processes embeddings **locally** using `sentence-transformers` with **Qdrant** vector store.

**No API token required** - all inference runs on local CPU/GPU.

**Model**: `all-MiniLM-L6-v2` (384 dimensions, ~90MB)
**Vector Store**: Qdrant (persistent, local)

In [ ]:
# Install dependencies
!pip install langchain-qdrant qdrant-client langchain-huggingface sentence-transformers -q

In [ ]:
import os
import numpy as np
from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

print("✅ All imports successful")

In [ ]:
# Load HuggingFaceEmbeddings (sentence-transformers) for local inference
model_name = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    encode_kwargs={"normalize_embeddings": True}
)

print(f"✅ Model loaded: {model_name}")

In [ ]:
# Create Qdrant client and vector store
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

persist_dir = "./qdrant_db"
collection_name = "nexlify_kb"

# Create Qdrant client
client = QdrantClient(path=persist_dir)

# Create collection if not exists
client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=384,  # all-MiniLM-L6-v2 dimension
        distance=Distance.COSINE
    )
)
print(f"✅ Collection '{collection_name}' ready")

In [ ]:
# Create LangChain Qdrant wrapper
vectorstore = QdrantVectorStore.from_existing_collection(
    client=client,
    collection_name=collection_name,
    embedding=embeddings,
    vector_name="dense"
)

print("✅ QdrantVectorStore ready")

In [ ]:
# Add documents to vector store
docs = [
    Document(
        page_content="Nexlify Corp reported record revenue of $2.4B in Q4 2025, up 45% year-over-year. AI datacenter chip sales drove the growth, accounting for 68% of total revenue.",
        metadata={"source": "earnings", "quarter": "Q4-2025", "type": "financial"}
    ),
    Document(
        page_content="H100 GPU supply constraints eased significantly in Q4 2025. This allowed Nexlify to increase production by 15%, reducing the order backlog from 8 months to 3 months.",
        metadata={"source": "supply_chain", "quarter": "Q4-2025", "type": "operations"}
    ),
    Document(
        page_content="Nexlify AI Accelerator NXA-1 launched in Q4 2025, achieving 3x inference speed compared to NVIDIA H100 while consuming 40% less power. First customers include major cloud providers.",
        metadata={"source": "product", "quarter": "Q4-2025", "type": "product"}
    ),
    Document(
        page_content="Board of Directors approved $500M stock buyback program for Q1 2026, reflecting confidence in the company's growth trajectory and strong cash position of $3.2B.",
        metadata={"source": "board", "quarter": "Q4-2025", "type": "corporate"}
    ),
    Document(
        page_content="AI datacenter chip market expected to grow 35% CAGR through 2028, reaching $85B. Nexlify positioned to capture 25% market share with expanded product portfolio.",
        metadata={"source": "market_analysis", "quarter": "Q4-2025", "type": "market"}
    ),
]

vectorstore.add_documents(docs)
print(f"✅ Added {len(docs)} documents to vector store")

# Count total embedded
info = client.get_collection(collection_name)
print(f"Total documents in store: {info.points_count}")

In [ ]:
# Query the vector store
query = "Q4 2025 revenue growth and financial performance"

# Semantic similarity search
results = vectorstore.similarity_search(query, k=2)

print(f"Query: '{query}'\n")
print("Top 2 results:")
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Content: {doc.page_content[:150]}...")
    print(f"Metadata: {doc.metadata}")

In [ ]:
# Query with similarity scores
query = "What happened with GPU supply chain in Q4?"

results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

print(f"Query: '{query}'\n")
print("Results with similarity scores:")
for i, (doc, score) in enumerate(results_with_scores):
    print(f"\n--- Result {i+1} (score: {score:.4f}) ---")
    print(f"Content: {doc.page_content}")

In [ ]:
# Use as a retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}
)

query = "revenue and earnings"
docs = retriever.invoke(query)

print(f"Query: '{query}'")
for i, doc in enumerate(docs):
    print(f"\n{i+1}. {doc.page_content[:100]}...")

In [ ]:
# Benchmark embedding speed
import time

benchmark_texts = [
    "AI datacenter chip market expected to grow 35% CAGR through 2028",
] * 100

start = time.time()
embeddings_batch = embeddings.embed_documents(benchmark_texts)
elapsed = time.time() - start

print(f"Embedded {len(benchmark_texts)} documents in {elapsed:.3f}s")
print(f"Speed: {len(benchmark_texts) / elapsed:.1f} docs/sec")
print(f"Latency: {elapsed / len(benchmark_texts) * 1000:.2f} ms/doc")

## Summary

### Tech Stack
- **Embeddings**: `HuggingFaceEmbeddings` (sentence-transformers/all-MiniLM-L6-v2)
- **Vector Store**: Qdrant (local, persistent)
- **Wrapper**: LangChain Qdrant integration

### Key Code
```python
# Create Qdrant collection
client = QdrantClient(path="./qdrant_db")
client.create_collection(
    collection_name="nexlify_kb",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

# LangChain wrapper
vectorstore = QdrantVectorStore.from_existing_collection(
    client=client,
    collection_name="nexlify_kb",
    embedding=embeddings
)
```

**One-time cost**: ~90MB model download
**Ongoing**: No API costs, no rate limits, fully offline capable